# C2 · Locked evaluation

Dùng dev để khóa thresholds, stress radii và checkpoint hashes. Xem báo cáo dev và kiểm tra matched budget trước khi chạy cell test. Bootstrap intervals là exploratory ở rare-event regimes. Trục dọc trong frontier cho feasible lower và numerical upper của nghiệm tối ưu trên finite graph.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
REPO = DRIVE / "C2_Base"
WORK = DRIVE / "C2_work"
DATA_ROOT = Path("/content/digiface72")  # Extracted, authorized 72-capture partition
MANIFEST = WORK / "manifests/digiface72.csv"

assert (REPO / "configs/pilot.yaml").is_file(), f"Set REPO to the copied C2_Base source folder: {REPO}"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")

def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], cwd=REPO, check=True)

In [ ]:
import torch

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SCOPE = "pilot"
SEEDS = [17]
LEVELS = [0.01, 0.001]
RADII = [0.1, 0.25, 0.5]
BOOTSTRAP_REPS = 500

def paths_for(seed):
    run_root = WORK / "runs" / SCOPE / f"seed-{seed}"
    return run_root, run_root / "evaluation"

def evaluate(action, run_root, evaluation, *extra):
    run("scripts.evaluate", action, "--run-root", run_root, "--output", evaluation,
        "--manifest", MANIFEST, "--data-root", DATA_ROOT, "--device", DEVICE, *extra)

In [ ]:
for seed in SEEDS:
    run_root, evaluation = paths_for(seed)
    lock = evaluation / "locked_protocol.json"
    if not lock.exists():
        evaluate("lock", run_root, evaluation, "--levels", *LEVELS,
                 "--radii", *RADII, "--bootstrap-reps", BOOTSTRAP_REPS)
    plan = json.loads(lock.read_text(encoding="utf-8"))
    dev = json.loads((evaluation / "dev.json").read_text(encoding="utf-8"))
    print(seed, plan["skipped_updates"], plan["operating_points"])
    for level, result in dev["hard"].items():
        print(level, result["ordinary"]["nominal"]["arms"])

In [ ]:
RUN_TEST = False  # Set True after completing the dev gate and locking the protocol.
RUN_OPTIONAL_CAL = False
test_reports = []
for seed in SEEDS:
    run_root, evaluation = paths_for(seed)
    if RUN_OPTIONAL_CAL:
        evaluate("cal", run_root, evaluation)
    if RUN_TEST:
        evaluate("test", run_root, evaluation)
        test_reports.append(evaluation / "test.json")
if len(test_reports) > 1:
    run("scripts.report_runs", "seeds", "--reports", *test_reports,
        "--output", WORK / "runs" / SCOPE / "seed_summary.json")

In [ ]:
import matplotlib.pyplot as plt

seed = SEEDS[0]
_, evaluation = paths_for(seed)
split_to_plot = "test" if RUN_TEST else "dev"
report = json.loads((evaluation / f"{split_to_plot}.json").read_text(encoding="utf-8"))
level, radius = str(LEVELS[0]), str(RADII[0])
hard = report["hard"][level]
fig, ax = plt.subplots(figsize=(6, 4))
for arm, bracket in hard["frontier"][radius]["brackets"].items():
    if bracket["status"] == "PASS":
        tar = hard["ordinary"]["nominal"]["arms"][arm]["tar"]
        ax.plot([tar, tar], [bracket["feasible_lower"], bracket["relaxed_upper"]],
                "o-", label=arm)
ax.set(xlabel="Nominal TAR", ylabel="Worst-case hard FMR bracket",
       title=f"{split_to_plot} · seed {seed} · target dev FMR {level} · radius {radius}")
if ax.lines:
    ax.legend()
plt.show()